# STEP-Risk vs O'Brien (STS) — AUROC comparison & statistical testing

Reproduces the revised analysis requested by the reviewer:

1. **Master AUROC table** with 95% bootstrap CI — each model scored on the patients it can serve.
2. **Pairwise difference tests** — unpaired bootstrap of ΔAUROC (STEP-Risk − O'Brien), each model on its full applicable population, **one-sided** (H1: STEP-Risk > O'Brien).
3. **Benjamini–Hochberg FDR correction applied jointly to all 48 one-sided p-values** (2 STEP-Risk variants × 3 cohorts × 8 outcomes).
4. **Forest plots** of ΔAUROC with 95% CI, post-surgery and pre-op.

**Method decisions**
- **Full applicable population.** STEP-Risk applies to the whole cohort; O'Brien is defined only for the 3 major cardiac procedures it was developed on. Each model is evaluated on its own applicable population and the two are resampled independently (unpaired bootstrap of the AUROC difference).
- **One-sided test.** The hypothesis is directional (STEP-Risk discriminates better than O'Brien): `p_one = mean(ΔAUROC_boot ≤ 0)`.
- **Single BH family of 48 tests.** The 24 pre-op-vs-O'Brien and 24 post-surgery-vs-O'Brien p-values are pooled and corrected together; `q_BH` is reported and significance (`sig`) is called at q < 0.05.
- The two-sided 95% percentile CI of ΔAUROC is reported as the interval estimate for the forest plots.

Models: **post** = STEP-Risk post-surgery, **pre** = STEP-Risk pre-operative, **sts** = O'Brien/STS score.
Seed = 42, 10,000 bootstrap replicates throughout.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests

# ---- paths ----
BASE = 'Risk_scores_and_true_outcome'   # folder with the risk-score + outcome CSVs
OUT  = 'analysis_outputs'               # where tables/figures are written
os.makedirs(OUT, exist_ok=True)

N_BOOT = 10000
SEED   = 42
ALPHA  = 0.05

AEnames  = ['mortality','stroke','renal','cpvntlng','reop','major','PLOS','SLOS']
STS_COLS = ['predmort','predstro','predrenf','predvent','predreop','predmm','pred14d','pred6d']
COHORTS  = ['retrospective_MMC','prospective_MMC','external_BWH']

COH_LABEL = {'retrospective_MMC':'MMC (retro)',
             'prospective_MMC':'MMC (temporal)',
             'external_BWH':'External (BWH)'}

MODEL_LABEL = {'post':'STEP-Risk (post-surgery)','pre':'STEP-Risk (pre-op)','sts':"O'Brien (STS)"}

OUT_LABEL = {'mortality':'Mortality','stroke':'Stroke','renal':'Renal failure',
             'cpvntlng':'Prolonged ventilation','reop':'Reoperation','major':'Major morbidity',
             'PLOS':'Prolonged LOS','SLOS':'Short LOS'}

## 1. Load the three cohorts

Each cohort has: true outcomes, STEP-Risk post-surgery probs, STEP-Risk pre-op probs, and O'Brien/STS probs. The STS columns are renamed to the 8 outcome names in a fixed order.

In [ ]:
def load_cohort(which):
    if which == 'external_BWH':
        outcomes = pd.read_csv(f'{BASE}/BWH_outcomes.csv', index_col=0).iloc[:, 1:]
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg_BWH_pros.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features_BWH_pros.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/BWH_stspred.csv', index_col=0)
    elif which == 'retrospective_MMC':
        outcomes = pd.read_csv(f'{BASE}/internal_Ytest.csv', index_col=0).reset_index(drop=True)
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/retrospective_MMC_stspred_test.csv'); sts.columns = sts.columns.str.lower()
    elif which == 'prospective_MMC':
        outcomes = pd.read_csv(f'{BASE}/pros_outcomes.csv', index_col=0).reset_index(drop=True)
        RF  = pd.read_csv(f'{BASE}/Result_probs_topn_features_postsurg_pros.csv', index_col=0)
        pre = pd.read_csv(f'{BASE}/Result_probs_preop_features_pros.csv', index_col=0)
        sts = pd.read_csv(f'{BASE}/prospective_MMC_stspred.csv'); sts.columns = sts.columns.str.lower()
    else:
        raise ValueError(which)
    sts = sts[STS_COLS].copy(); sts.columns = AEnames
    return (outcomes.reset_index(drop=True), RF.reset_index(drop=True),
            pre.reset_index(drop=True), sts.reset_index(drop=True))

DATA = {w: load_cohort(w) for w in COHORTS}
MODEL_SRC = {'post': 1, 'pre': 2, 'sts': 3}   # index into the DATA tuple (o, RF, pre, sts)

# sanity: N and STS coverage
for w in COHORTS:
    o, RF, pre, sts = DATA[w]
    print(f'{w:20s} N={len(o):6d}  STS-available={int(sts[AEnames[0]].notna().sum()):6d}')

## 2. AUC helper

`fast_auc` is a vectorised Mann–Whitney AUC (tie-corrected) — identical to `sklearn.roc_auc_score` but ~20× faster inside a 10,000-iteration bootstrap.

In [ ]:
def fast_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    n1 = y_true.sum(); n0 = len(y_true) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(y_score)
    return (r[y_true == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# equivalence check vs sklearn
_o, _RF, _pre, _sts = DATA['external_BWH']
for AE in ['mortality','major']:
    y = _o[AE].values.astype(float); s = _RF[AE].values.astype(float)
    assert abs(fast_auc(y, s) - roc_auc_score(y, s)) < 1e-9
print('fast_auc matches sklearn roc_auc_score')

## 3. Bootstrap functions

- `auc_ci` — percentile 95% CI for a single model's AUROC, resampling its own non-NaN patients.
- `unpaired_boot_compare` — resamples each model **independently on its own applicable population**; the ΔAUROC CI and the one-sided p-value come from the bootstrap distribution of (AUC_a − AUC_b).

`p_one` = one-sided p-value for H1: model_a > model_b = mean(ΔAUROC_boot ≤ 0).

In [ ]:
def auc_ci(y_true, y_prob, n_boot=N_BOOT, seed=SEED):
    y_true = np.asarray(y_true, float); y_prob = np.asarray(y_prob, float)
    m = ~(np.isnan(y_true) | np.isnan(y_prob))
    y_true, y_prob = y_true[m], y_prob[m]
    n = len(y_true); npos = int(y_true.sum())
    auc = fast_auc(y_true, y_prob)
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = fast_auc(y_true[idx], y_prob[idx])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return dict(auc=auc, lo=lo, hi=hi, n=n, npos=npos)


def unpaired_boot_compare(y_true, prob_a, prob_b, n_boot=N_BOOT, seed=SEED):
    y = np.asarray(y_true, float); a = np.asarray(prob_a, float); b = np.asarray(prob_b, float)
    ma = ~(np.isnan(y) | np.isnan(a)); mb = ~(np.isnan(y) | np.isnan(b))
    ya, aa = y[ma], a[ma]; yb, bb = y[mb], b[mb]
    auc_a, auc_b = fast_auc(ya, aa), fast_auc(yb, bb)
    na, nb = len(ya), len(yb)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        ia = rng.integers(0, na, na); ib = rng.integers(0, nb, nb)
        diffs[k] = fast_auc(ya[ia], aa[ia]) - fast_auc(yb[ib], bb[ib])
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    p_one = float(np.mean(diffs <= 0))          # H1: model_a > model_b
    return dict(auc_a=auc_a, auc_b=auc_b, n_a=na, n_b=nb,
                npos_a=int(ya.sum()), npos_b=int(yb.sum()),
                mean_diff=auc_a-auc_b, ci_lo=lo, ci_hi=hi, p_one=p_one)

## 4. Master AUROC table (per cohort × outcome × model)

Each model's AUROC with a 95% bootstrap CI, computed on that model's own non-NaN patients. `n` / `npos` show the applicable-population size and event count.

In [ ]:
rows = []
for w in COHORTS:
    tup = DATA[w]; o = tup[0]
    for AE in AEnames:
        y = o[AE].values.astype(float)
        for mkey, si in MODEL_SRC.items():
            prob = tup[si][AE].values.astype(float)
            r = auc_ci(y, prob)
            rows.append(dict(cohort=w, outcome=AE, model=mkey,
                             auc=r['auc'], lo=r['lo'], hi=r['hi'], n=r['n'], npos=r['npos']))
auc_df = pd.DataFrame(rows)
auc_df['AUROC (95% CI)'] = auc_df.apply(lambda r: f"{r.auc:.3f} ({r.lo:.3f}–{r.hi:.3f})", axis=1)
auc_df.to_csv(f'{OUT}/auroc_master_table.csv', index=False)

# wide view: rows = cohort × outcome (fixed order), cols = the three models
wide = (auc_df.assign(model=auc_df.model.map({'pre':'STEP-Risk preop','post':'STEP-Risk postsurg','sts':"O'Brien (STS)"}),
                      cohort=pd.Categorical(auc_df.cohort, COHORTS, ordered=True),
                      outcome=pd.Categorical(auc_df.outcome, AEnames, ordered=True))
        .pivot_table(index=['cohort','outcome'], columns='model', values='AUROC (95% CI)',
                     aggfunc='first', observed=True)
        .sort_index())
wide = wide[["O'Brien (STS)", 'STEP-Risk preop', 'STEP-Risk postsurg']]
wide.to_csv(f'{OUT}/auroc_table_wide.csv')
wide

## 5. Pairwise comparisons — one-sided unpaired bootstrap, 48 tests

Two comparisons per cohort × outcome: (post vs sts) and (pre vs sts) → 2 × 3 × 8 = 48 tests.

In [ ]:
PAIRS = [('post','sts'), ('pre','sts')]

unp_rows = []
for w in COHORTS:
    tup = DATA[w]; o = tup[0]
    for AE in AEnames:
        y = o[AE].values.astype(float)
        for ma, mb in PAIRS:
            pa = tup[MODEL_SRC[ma]][AE].values.astype(float)
            pb = tup[MODEL_SRC[mb]][AE].values.astype(float)
            u = unpaired_boot_compare(y, pa, pb)
            u.update(cohort=w, outcome=AE, model_a=ma, model_b=mb)
            unp_rows.append(u)
unp_df = pd.DataFrame(unp_rows)
assert len(unp_df) == 48, len(unp_df)
print('tests:', len(unp_df))

## 6. Benjamini–Hochberg correction over all 48 tests

The 48 one-sided p-values (24 post-surgery vs O'Brien + 24 pre-op vs O'Brien) form a **single** BH family. `q_BH` is the adjusted p-value; `sig` is called at q < 0.05.

In [ ]:
rej, q, _, _ = multipletests(unp_df['p_one'], alpha=ALPHA, method='fdr_bh')
unp_df['q_BH'] = q
unp_df['sig']  = rej

print('raw p_one < 0.05 : %d/48' % (unp_df.p_one < ALPHA).sum())
print('BH q     < 0.05 : %d/48' % unp_df.sig.sum())
unp_df.to_csv(f'{OUT}/diff_all48_unpaired_onesided_BH.csv', index=False)

## 7. Final comparison tables (STEP-Risk vs O'Brien)

`build_final(model_a)` formats the 24 rows for one STEP-Risk variant vs O'Brien. The `q_BH` values come from the joint 48-test correction above — they are **not** recomputed per table.

In [ ]:
def build_final(model_a):
    m = unp_df[(unp_df.model_a==model_a)&(unp_df.model_b=='sts')].copy()
    out = pd.DataFrame({
        'Cohort':  m.cohort.map(COH_LABEL),
        'Outcome': m.outcome.map(OUT_LABEL),
        'N_STEP':  m.n_a, 'N_OBrien': m.n_b,
        'Events_STEP': m.npos_a, 'Events_OBrien': m.npos_b,
        'AUROC_STEP':   m.auc_a.round(3),
        'AUROC_OBrien': m.auc_b.round(3),
        'dAUROC':  m.mean_diff.round(3),
        'CI95':    m.apply(lambda r: f"{r.ci_lo:.3f}–{r.ci_hi:.3f}", axis=1),
        'p_one':   m.p_one,
        'q_BH':    m.q_BH,
        'sig':     m.sig.map({True:'*', False:''}),
    })
    out['Cohort']  = pd.Categorical(out['Cohort'],  [COH_LABEL[c] for c in COHORTS], ordered=True)
    out['Outcome'] = pd.Categorical(out['Outcome'], [OUT_LABEL[a] for a in AEnames], ordered=True)
    return out.sort_values(['Cohort','Outcome']).reset_index(drop=True)

final_post = build_final('post')
final_pre  = build_final('pre')
final_post.to_csv(f'{OUT}/diff_post_vs_obrien_FINAL.csv', index=False)
final_pre.to_csv(f'{OUT}/diff_pre_vs_obrien_FINAL.csv', index=False)
print('post-surgery: %d/24 significant after BH (q < 0.05)' % (final_post.sig=='*').sum())
print('pre-op:       %d/24 significant after BH (q < 0.05)' % (final_pre.sig=='*').sum())
final_post

In [ ]:
final_pre